# 06 · One national model or one per ecoregion?

**Question:** the platform fits one risk model for all of Guinea. Guinea spans five ecoregions, from the West Sudanian savanna to the lowland forests and mangroves (RESOLVE 2017; Dinerstein et al. 2017), and stage 1 reports by ecoregion. Would one model per ecoregion predict the loss better?

The drivers suggest not: shifting cultivation causes 82–86 % of the loss in every forest ecoregion, and the savanna, where permanent agriculture dominates, holds under 2 % of it (`studies/forest_loss_drivers.ipynb`, section 5). This notebook tests it two ways, with the improved model trained on the shares (notebook 04):

* **Ranking:** AUC over the 30 m forest pixels, nationally and inside each ecoregion, on a fixed sample of 3 million test cells (`scripts/model_variants.py`).
* **Verra's tests:** fit and prediction MedAE nationally, the loss allocated to each ecoregion against the loss it had, and the error over the ~100,000 ha blocks mostly inside each ecoregion, on full risk maps (`scripts/lean_and_ecoregions.py`).

Each ecoregion's model is trained on its own cells and put back on one scale with its loss rate in the calibration window, so that its risks can be compared with the others' when the loss is allocated. **The bar:** a change counts when it moves AUC the same way in all three windows and by more than twice the run-to-run noise (seed SD 0.0017) on average.

Results are saved in `outputs/runs/`; set `RERUN = True` to compute them again (about 15–20 minutes per window for each script).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "scripts"))
import model as MM
import model_variants as D
import lean_and_ecoregions as F

OUT = HERE.parent / "outputs" / "summary"
RERUN = False
variants = [MM.cached(D.split_name(s), lambda s=s: D.run_split(s), rerun=RERUN) for s in MM.SPLITS]
followups = [MM.cached(F.split_name(s), lambda s=s: F.run_split(s), rerun=RERUN) for s in MM.SPLITS]
WINDOWS = [run["split"].split(" → ")[1] for run in variants]
NOISE = 0.0017

## 1 · Ranking the loss

The national model against one model per forest ecoregion, on the same test cells (raw: the ecoregions' risks as they come; rescaled: put back on one scale).

In [2]:
rows = []
for w, run in zip(WINDOWS, variants):
    e = run["ecoregions"]
    rows.append({"Test window": w, "National": np.mean(e["national"]), "Per ecoregion, raw": np.mean(e["per_ecoregion_raw"]),
                 "Per ecoregion, rescaled": np.mean(e["per_ecoregion_corrected"])})
eco = pd.DataFrame(rows).set_index("Test window")
eco["Rescaled − national"] = eco["Per ecoregion, rescaled"] - eco["National"]
eco.round(4).to_csv(OUT / "06_ecoregion_ranking.csv")
print(f"Bar: a mean change of {2 * NOISE:.4f}")
eco.round(4)

Bar: a mean change of 0.0034


,National,"Per ecoregion, raw","Per ecoregion, rescaled",Rescaled − national
Test window,,,,
2017–2019,0.7579,0.7593,0.7604,0.0025
2020–2022,0.7634,0.7622,0.7641,0.0007
2022–2024,0.7374,0.7407,0.7400,0.0026


In [3]:
inside = pd.DataFrame({w: {n: np.mean(v["corrected"]) - np.mean(v["national"]) for n, v in run["ecoregions"]["inside"].items()}
                       for w, run in zip(WINDOWS, variants)})
inside["Mean"] = inside.mean(axis=1)
print("AUC inside each ecoregion: per-ecoregion model (rescaled) minus national model")
inside.round(4)

AUC inside each ecoregion: per-ecoregion model (rescaled) minus national model


,2017–2019,2020–2022,2022–2024,Mean
Guinean forest-savanna,-0.0001,0.0003,0.0023,0.0008
Guinean mangroves,0.0254,0.0123,0.0276,0.0218
Guinean montane forests,0.0089,0.0056,0.0026,0.0057
West Sudanian savanna,0.0285,0.0142,0.0096,0.0174
Western Guinean lowland forests,0.0111,0.0082,0.0039,0.0077


**Reading:**

* **Nationally, one model per ecoregion ranks the loss better in every window, but by little:** +0.0025, +0.0007 and +0.0026, a mean of +0.0019, under the bar of 0.0034.
* **Inside each ecoregion the gain is larger**, most in the small ecoregions: mangroves +0.022, West Sudanian savanna +0.017, lowland forests +0.008, montane forests +0.006. The forest-savanna, which holds most of the forest, gains nothing. The national model ranks well between ecoregions, which is most of the national AUC, but less well within the small ones, where its training draws few cells.

## 2 · Verra's tests

Both with the same allocation (risk classes within the admin regions, VT0007's modelling regions).

In [4]:
rows = []
for w, run in zip(WINDOWS, followups):
    for label, name in (("national", "National model"), ("per_ecoregion", "One model per ecoregion")):
        r = run["ecoregion"][label]
        rows.append({"Test window": w, "Model": name, "AUC": np.mean(r["aucs"]),
                     "Fit MedAE (ha)": r["fit_medae_ha"], "Prediction MedAE (ha)": r["prediction_medae_ha"]})
national = pd.DataFrame(rows)
national.round(4).to_csv(OUT / "06_ecoregion_verra.csv", index=False)
display(national.set_index(["Test window", "Model"]).round(3))
national.groupby("Model")[["AUC", "Fit MedAE (ha)", "Prediction MedAE (ha)"]].mean().round(3)

AUC  Fit MedAE (ha)  \
Test window Model                                            
2017–2019   National model           0.757         107.716   
            One model per ecoregion  0.760         104.897   
2020–2022   National model           0.763         149.596   
            One model per ecoregion  0.764         107.674   
2022–2024   National model           0.737         127.629   
            One model per ecoregion  0.739          95.422   

                                     Prediction MedAE (ha)  
Test window Model                                           
2017–2019   National model                         233.938  
            One model per ecoregion                296.052  
2020–2022   National model                         167.520  
            One model per ecoregion                187.859  
2022–2024   National model                         137.121  
            One model per ecoregion                163.467

,AUC,Fit MedAE (ha),Prediction MedAE (ha)
Model,,,
National model,0.752,128.314,179.526
One model per ecoregion,0.754,102.664,215.792


In [5]:
rows = []
for w, run in zip(WINDOWS, followups):
    for label, name in (("national", "National"), ("per_ecoregion", "Per ecoregion")):
        r = run["ecoregion"][label]
        for e, (predicted, observed) in r["loss_by_ecoregion"].items():
            m = r["medae_by_ecoregion"][e]
            rows.append({"Test window": w, "Ecoregion": e, "Model": name,
                         "Allocated / observed loss (%)": 100 * (predicted / observed - 1),
                         "Blocks": m["blocks"], "Fit MedAE (ha)": m["fit"], "Prediction MedAE (ha)": m["prediction"]})
by_eco = pd.DataFrame(rows)
by_eco.round(2).to_csv(OUT / "06_by_ecoregion.csv", index=False)
mean = by_eco.groupby(["Ecoregion", "Model"]).agg(**{
    "Blocks": ("Blocks", "first"),
    "Allocated vs observed loss (%), mean": ("Allocated / observed loss (%)", "mean"),
    "Largest miss (%)": ("Allocated / observed loss (%)", lambda v: v.abs().max()),
    "Fit MedAE (ha)": ("Fit MedAE (ha)", "mean"), "Prediction MedAE (ha)": ("Prediction MedAE (ha)", "mean")})
print("Mean of the three windows. Blocks: ~100,000 ha blocks mostly inside the ecoregion (none for the mangroves)")
mean.round(1)

Mean of the three windows. Blocks: ~100,000 ha blocks mostly inside the ecoregion (none for the mangroves)


Blocks  \
Ecoregion                       Model                   
Guinean forest-savanna          National          109   
                                Per ecoregion     109   
Guinean mangroves               National            0   
                                Per ecoregion       0   
Guinean montane forests         National           25   
                                Per ecoregion      25   
West Sudanian savanna           National           38   
                                Per ecoregion      38   
Western Guinean lowland forests National            9   
                                Per ecoregion       9   

                                               Allocated vs observed loss (%), mean  \
Ecoregion                       Model                                                 
Guinean forest-savanna          National                                        2.8   
                                Per ecoregion                                   4.2   
Guinean mangroves               National                                      -11.3   
                                Per ecoregion                                 -37.7   
Guinean montane forests         National                                       -5.8   
                                Per ecoregion                                  -6.0   
West Sudanian savanna           National                                      -15.1   
                                Per ecoregion                                 -15.1   
Western Guinean lowland forests National                                       -1.8   
                                Per ecoregion                                  -4.1   

                                               Largest miss (%)  \
Ecoregion                       Model                             
Guinean forest-savanna          National                    5.4   
                                Per ecoregion               7.1   
Guinean mangroves               National                   25.8   
                                Per ecoregion              58.4   
Guinean montane forests         National                   10.1   
                                Per ecoregion              11.6   
West Sudanian savanna           National                   38.2   
                                Per ecoregion              34.2   
Western Guinean lowland forests National                   11.2   
                                Per ecoregion              10.8   

                                               Fit MedAE (ha)  \
Ecoregion                       Model                           
Guinean forest-savanna          National                192.9   
                                Per ecoregion           180.9   
Guinean mangroves               National                  NaN   
                                Per ecoregion             NaN   
Guinean montane forests         National                142.5   
                                Per ecoregion            98.5   
West Sudanian savanna           National                  1.6   
                                Per ecoregion             1.5   
Western Guinean lowland forests National                510.1   
                                Per ecoregion           287.5   

                                               Prediction MedAE (ha)  
Ecoregion                       Model                                 
Guinean forest-savanna          National                       273.5  
                                Per ecoregion                  312.5  
Guinean mangroves               National                         NaN  
                                Per ecoregion                    NaN  
Guinean montane forests         National                       214.9  
                                Per ecoregion                  203.3  
West Sudanian savanna           National                         3.1  
                                Per ecoregion                    2.7  
Western Guinean lowla

**Reading:**

* **Per-ecoregion models fit the calibration window better but predict the next window worse, in every window.** Fit MedAE: 103 against 128 ha. Prediction MedAE: 216 against 180 ha. Each model is rescaled with its ecoregion's loss rate in the calibration window, which matches that window and carries it into the next, where the rates have moved.
* **By ecoregion, they help where the national model is weakest and hurt where most of the forest is.** Prediction errors fall in the montane forests (203 against 215 ha) and the lowland forests (632 against 695). They rise in the forest-savanna (313 against 274), which holds 109 of the 181 blocks with forest, so the national median goes up.
* **The national model allocates each main ecoregion's loss within 6 % on average:** forest-savanna +3 %, lowland forests −2 %, montane forests −6 %. It allocates too little in the West Sudanian savanna (−15 %, up to −38 % in a window) and the mangroves (−11 %). Per-ecoregion models do no better on these, and much worse in the mangroves (−38 %).

## 3 · What this means

* **Keep one national model.** Per-ecoregion models rank the loss better inside the small ecoregions and fit the calibration window better, but predict the next window worse nationally, in every window. Verra's prediction test is the one that matters for the baseline. This agrees with the drivers: the same clearing process dominates every forest ecoregion.
* **Reporting by ecoregion works with the national model**, within about 6 % of each main forest ecoregion's loss. The West Sudanian savanna and the mangroves are under-allocated (−15 % and −11 %), which is worth stating when ecoregion figures are reported.
* **The platform needs no change here:** it already fits one national model.
* **Next, notebook 07:** the improved model against the deployed one.